# Barriers

In [ ]:
import osmnx as ox
import cityImage as ci
import pandas as pd
import geopandas as gpd
from matplotlib.colors import LinearSegmentedColormap

%matplotlib inline

import warnings
warnings.simplefilter(action="ignore")

### New API helper functions
Raw file/OSM acquisition is delegated to GeoPandas/OSMnx; cityImage keeps schema conversion and Lynchian/image-element semantics.

In [ ]:
# initialise path, names, etc.
city_name = 'Paris'
epsg = 27571
crs = 'EPSG:'+str(epsg)
place = 'Paris, Ile-de-France'

## Barriers Identification

Downloading from OSM

Choose between the following methods:

* `OSMplace`, providing an OSM place name (e.g. City).
* `polygon`, providing a Polygon (coordinates must be in units of latitude-longitude degrees).
* `distance_from_address`, providing a precise address and setting the `distance` parameter.
* `distance_from_point`, providing point coordinates (in units of latitude-longitude degrees) and setting the `distance` parameter to build the bounding box around the point.

In [ ]:
download_method = 'OSMplace'

In [ ]:
barriers = ci.barriers_from_osm(
    place,
    download_method=download_method,
    crs=crs,
    include_primary=True,
    include_secondary=False,
    parks_min_area=100000,
)
barriers.reset_index(inplace=True, drop=True)
barriers["barrierID"] = barriers.index.astype(int)

In [ ]:
# Barriers are already combined by ci.barriers_from_osm().

**Visualisation**

In [ ]:
barriers.sort_values(by = 'barrier_type', ascending = False, inplace = True)  
colors = ['green', 'red', 'gray', 'blue']

cmap = LinearSegmentedColormap.from_list('cmap', colors, N=len(colors))
fig = ci.plot_gdf(gdf = barriers, column = 'barrier_type', black_background = False, title = city_name+': Barriers', 
                  legend = True, cmap = cmap)

## Incorporating Barriers into the Street Network
This is an optional step that allows modelling the effect of barriers on pedestrian movement, for exmaple, in agent-based modelling, or in route-choice modelling.

**Download the street network**

Choose between the following methods:

* `OSMplace`, providing an OSM place name (e.g. City).
* `polygon`, providing a Polygon (coordinates must be in units of latitude-longitude degrees).
* `distance_from_address`, providing a precise address and setting the `distance` parameter.
* `distance_from_point`, providing point coordinates (in units of latitude-longitude degrees) and setting the `distance` parameter to build the bounding box around the point.

Downloading the graph and cleaning it (see the notebook *01-Nodes_Paths_fromOSM* for details on the cleaning process)

In [ ]:
nodes_graph, edges_graph = ci.network_from_osm(place, download_method=download_method, network_type="walk", crs=crs)
nodes_graph, edges_graph = ci.clean_network(nodes_graph, edges_graph, dead_ends = True, remove_islands = True,
                            self_loops = True, same_vertexes_edges = True)

**(or) Load it from local path**

In [ ]:
input_path = '../data/'+city_name
nodes_graph = gpd.read_file(input_path+'_nodes.gpkg')
edges_graph = gpd.read_file(input_path+'_edges.gpkg')

nodes_graph.index, edges_graph.index  = nodes_graph.nodeID, edges_graph.edgeID
nodes_graph.index.name, edges_graph.index.name  = None, None

**Visualisation**

In [ ]:
fig = ci.plot_gdf(edges_graph, black_background = False, geometry_size = 1.0, alpha = 1.0, 
                 color = 'black', title = city_name+': Street Network', figsize = (10,10))

### Assigning barriers to street segments
Type of Barriers:

* *Positive barriers*, from a pedestrian perspective: Waterbodies, Parks.
* *Negative barriers*, from a pedestrian perspective: Major Roads, Railway Structures.
* *Structuring barriers* - Barriers which structure and shape the image of the city: Waterbodies, Major roads, Railways.

In [ ]:
# clipping barriers to case study area
envelope = edges_graph.unary_union.envelope
barriers_within = barriers[barriers.intersects(edges_graph.unary_union.envelope)]

### Street Segments *Along* and *Within* Positive Barriers

In [ ]:
sindex = edges_graph.sindex
# rivers
edges_graph = ci.along_water(edges_graph, barriers_within)
# parks
edges_graph = ci.along_within_parks(edges_graph, barriers_within)
# altogheter
edges_graph['p_barr'] = edges_graph['a_rivers']+edges_graph['w_parks']
edges_graph['p_barr'] = edges_graph.apply(lambda row: list(set(row['p_barr'])), axis = 1)

### Street Segments *Along* Negative Barriers

In [ ]:
tmp = barriers_within[barriers_within['barrier_type'].isin(['railway', 'road'])]
edges_graph['n_barr'] = edges_graph.apply(lambda row: ci.barriers_along(row['edgeID'], edges_graph, tmp, sindex,
                                            offset = 25), axis = 1)

### Street Segments *Crossing* any kind of barrier but parks: Structuring Barriers

In [ ]:
edges_graph = ci.assign_structuring_barriers(edges_graph, barriers_within)

**Visualisation**

In [ ]:
# positive barriers
edges_graph['p_bool'] = edges_graph.apply(lambda row: len(row['p_barr']) > 0, axis = 1)
tmp = edges_graph[edges_graph.p_bool].copy()

# base map
base_map_dict = {'base_map_gdf': edges_graph, 'base_map_alpha' : 0.3, 'base_map_color' : 'black'}
fig = ci.plot_gdf(tmp, black_background = False, figsize = (8, 8), color = 'red', title = city_name+': Streets along parks and rivers', 
              legend = False, **base_map_dict)

In [ ]:
# negative barriers
edges_graph['n_bool'] = edges_graph.apply(lambda row: len(row['n_barr']) > 0, axis = 1)
tmp = edges_graph[edges_graph.n_bool].copy()

# base map
fig = ci.plot_gdf(tmp, black_background = False, figsize = (8, 8), color = 'red', title = city_name+': Streets along negative barriers',
              legend = False, **base_map_dict)

In [ ]:
# separating barriers
tmp = edges_graph[edges_graph.sep_barr].copy()
ci.plot_gdf(tmp, black_background = False, figsize = (8, 8),  color = 'red', 
            title = city_name+': Streets crossing structuring barriers: water, highways, railways',
             **base_map_dict)

**Exporting**

In [ ]:
# saving barriers_gdf
output_path = '../output/'+city_name
barriers.to_file(output_path+"_barriers.gpkg", driver='GPKG')

# converting list fields to string
to_convert = ['a_rivers', 'w_parks','n_barr', 'p_barr']
edges_graph_string = edges_graph.copy()
for column in to_convert: 
    edges_graph_string[column] = edges_graph_string[column].astype(str)

edges_graph_string.to_file(output_path+"_edges_ped.gpkg", driver='GPKG')
nodes_graph.to_file(output_path+'_nodes_ped.gpkg', driver='GPKG')